# Run SMITH-Agent panel evaluation

This notebook starts from real H5AD inputs and writes a fresh ranking, panel, evaluation and run manifest. [Open the editable source notebook on GitHub](https://github.com/fym0503/SMITH/blob/main/docs/source/tutorials/notebooks/agent_section/05_SMITH_Agent_Evaluation_source.ipynb).

## Set data and output locations

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

def find_repository(start):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "reproducibility").exists():
            return candidate
    raise RuntimeError("Run this notebook inside a SMITH repository checkout.")

ROOT = find_repository(Path.cwd().resolve())
DATA_ROOT = Path(os.environ.get("SMITH_TUTORIAL_DATA", "data/tutorials")).expanduser().resolve()
OUTPUT_ROOT = Path(os.environ.get("SMITH_TUTORIAL_OUTPUT", "outputs/tutorials")).expanduser().resolve()
CASE_OUTPUT = OUTPUT_ROOT / 'agent'
EPOCHS = int(os.environ.get("SMITH_TUTORIAL_EPOCHS", '1'))
DEVICE = os.environ.get("SMITH_TUTORIAL_DEVICE", 'cpu')
print("Input and output roots are configured. Set SMITH_TUTORIAL_DATA/OUTPUT to override them.")

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


## Check real input files

In [ ]:
inputs = ['agent/liver_merfish/adata_healthy_nucseq.h5ad', 'agent/liver_merfish/adata_healthy_merfish.h5ad', 'agent/references/PSC011_C1_visium.h5ad', 'agent/references/WSSS_F_IMMsp9838712_visium.h5ad']
input_rows = []
for relative in inputs:
    path = DATA_ROOT / relative
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}. Run scripts/download_tutorial_data.py first.")
    digest = sha256_file(path)
    input_rows.append({"file": relative, "bytes": path.stat().st_size, "sha256": digest})
display(pd.DataFrame(input_rows))


## Run the workflow

In [ ]:
command = [
    sys.executable, str(ROOT / 'reproducibility/workflows/agent/run_tutorial.py'),
    "--data-root", str(DATA_ROOT),
    "--output-dir", str(CASE_OUTPUT),
    "--device", DEVICE,
    "--epochs", str(EPOCHS),
    "--seed", "42",
] + ['--panel-size', '64', '--max-cells', '3000']
display_command = [
    "python", 'reproducibility/workflows/agent/run_tutorial.py', "--data-root", "data/tutorials",
    "--output-dir", "outputs/tutorials/agent", "--device", DEVICE,
    "--epochs", str(EPOCHS), "--seed", "42",
] + ['--panel-size', '64', '--max-cells', '3000']
print(" ".join(display_command))
completed = subprocess.run(command, cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
if completed.returncode:
    print(completed.stdout)
    raise subprocess.CalledProcessError(completed.returncode, command)
manifest = json.loads((CASE_OUTPUT / "run_manifest.json").read_text())
print("Completed:", manifest["workflow"], "->", "outputs/tutorials/agent/run_manifest.json")


## Inspect newly generated outputs

In [ ]:
panel_path = CASE_OUTPUT / 'aggregation/integrated_top_64_panel.tsv'
ranking_path = sorted(CASE_OUTPUT.glob('aggregation/integrated_panel_rank.tsv'))[-1]
metrics_path = CASE_OUTPUT / 'evaluation/metrics.tsv'
panel = pd.read_csv(panel_path, sep="\t" if panel_path.suffix == ".tsv" else ",")
ranking = pd.read_csv(ranking_path, sep="\t" if ranking_path.suffix == ".tsv" else ",")
metrics = pd.read_csv(metrics_path, sep="\t")
print("Generated panel:", panel_path.relative_to(CASE_OUTPUT))
display(panel.head(15))
print("Generated ranking:", ranking_path.relative_to(CASE_OUTPUT))
display(ranking.head(10))
display(metrics)


## Analyze this run

In [ ]:
metrics = metrics[metrics["metric"].isin(['cell_type_accuracy', 'spatial_pearson'])].copy()
plot_data = metrics.pivot_table(index='metric', columns='panel', values="value", aggfunc="mean")
ax = plot_data.plot(kind="bar", figsize=(10, 4.5))
ax.set_ylabel("value")
ax.set_title('Run SMITH-Agent panel evaluation')
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()


## Tutorial run versus manuscript run

The tutorial trains a source panel and two real spatial-reference panels before rank aggregation. The paper-scale run uses five references and more seeds. Probe filtering is a separate backend-dependent stage and no pass rate is fabricated here.